# ENTORNO DE PRUEBAS - AGENTE-IA

## Configuración del ambiente

In [5]:
# Instalación de paquetes específicos sin alterar PyTorch/CUDA de Colab
!pip install -q chromadb pypdf rank-bm25 langchain-text-splitters sentence-transformers pymupdf pandas openpyxl

import os
import gc
import re
import pandas as pd
import fitz  # PyMuPDF
import torch
from typing import List, Dict, Any
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from rank_bm25 import BM25Okapi
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb

# Configuración global
DATA_DIR = "./data"
CHROMA_DB_DIR = "./data/chroma_db"
COLLECTION_NAME = "recubrimientos_industriales"

EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Entorno restablecido con éxito. Dispositivo activo: {DEVICE.upper()}")

✓ Entorno restablecido con éxito. Dispositivo activo: CPU


## Proceso y extracción de contenido

In [6]:
import pandas as pd
import pypdf
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TOKENIZADOR TÉCNICO UNIFICADO
def tokenize_technical_text(text):
    return re.findall(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ0-9\-_/\.]+", text.lower())

# DICCIONARIO DE CONCEPTOS EQUIVALENTES (NO RELACIONADOS)
DICCIONARIO_EXPANSION = {
    "granallado": ["chorro abrasivo", "abrasivo", "sa2.5", "sspc-sp10"],
    "epoxi": ["epoxico", "epo"],
    "pelicula": ["espesor", "mils", "micras", "micronage"],
    "limpieza": ["preparacion", "acondicionamiento"]
}

def expandir_query(query):
    tokens = tokenize_technical_text(query)
    query_expandida = []

    for token in tokens:
        query_expandida.append(token)
        if token in DICCIONARIO_EXPANSION:
            query_expandida.extend(DICCIONARIO_EXPANSION[token])

    return " ".join(query_expandida)

# CARGA DE PDFs CON ENRIQUECIMIENTO CONTEXTUAL Y METADATOS COMPLETOS (IDs ÚNICOS CORREGIDOS)
def load_and_chunk_pdf(pdf_path, doc_type, ignore_top_pages=3, chunk_size=650, chunk_overlap=120):
    reader = pypdf.PdfReader(pdf_path)
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    chunks = []
    metadatos = []
    chunk_counter = 0

    # Creamos un prefijo único a partir del nombre del tipo de documento (ej. 'ficha_tecnica' -> 'ficha')
    prefix = doc_type.lower().replace(" ", "_")

    for idx in range(ignore_top_pages, len(reader.pages)):
        page_text = reader.pages[idx].extract_text()
        if not page_text or len(page_text.strip()) < 50:
            continue

        page_chunks = text_splitter.split_text(page_text)

        for c in page_chunks:
            chunk_counter += 1
            # ID Único: ej. 'ficha_tecnica_1', 'manual_operativo_1'
            unique_chunk_id = f"{prefix}_{chunk_counter}"

            text_con_contexto = f"Documento: {doc_type} | Página: {idx + 1}\nContenido:\n{c}"

            chunks.append(text_con_contexto)
            metadatos.append({
                "chunk_id": unique_chunk_id,
                "fuente": pdf_path,
                "tipo_documento": doc_type,
                "pagina": idx + 1,
                "texto_raw": c
            })

    return chunks, metadatos

# CARGA DE EXCEL CON METADATOS
def load_excel_as_chunks(excel_path, entity_type):
    xls = pd.ExcelFile(excel_path)
    df = pd.read_excel(excel_path, sheet_name=xls.sheet_names[0])

    chunks = []
    metadatos = []
    chunk_counter = 0

    for idx, row in df.iterrows():
        chunk_counter += 1
        row_str_list = [f"{col}: {row[col]}" for col in df.columns if pd.notna(row[col])]
        row_text = f"Ficha de {entity_type} [{row.get('Código ' + entity_type, idx+1)}]:\n" + " | ".join(row_str_list)

        chunks.append(row_text)
        metadatos.append({
            "chunk_id": f"excel_{entity_type.lower()}_{chunk_counter}",
            "fuente": excel_path,
            "tipo_documento": f"Excel {entity_type}",
            "pagina": "N/A",
            "codigo": str(row.get('Código ' + entity_type, idx+1)),
            "texto_raw": row_text
        })

    return chunks, metadatos

print("Ejecutando Ingestión y Chunking...")
chunks_fichas, meta_fichas = load_and_chunk_pdf("Fichas_Tecnicas_Materiales_RIS.pdf", "Ficha Técnica", ignore_top_pages=1)
chunks_manuales, meta_manuales = load_and_chunk_pdf("Manuales_Procesos_Procedimientos_RIS.pdf", "Manual Operativo", ignore_top_pages=3)
chunks_mat_excel, meta_mat_excel = load_excel_as_chunks("Especificaciones_Materiales_RIS.xlsx", "Material")
chunks_eq_excel, meta_eq_excel = load_excel_as_chunks("Especificaciones_Equipos_RIS.xlsx", "Equipo")

todos_los_chunks = chunks_fichas + chunks_manuales + chunks_mat_excel + chunks_eq_excel
todos_los_metadatos = meta_fichas + meta_manuales + meta_mat_excel + meta_eq_excel
todos_los_ids = [m["chunk_id"] for m in todos_los_metadatos]

print(f"Total de fragmentos contextualizados: {len(todos_los_chunks)}")

Ejecutando Ingestión y Chunking...
Total de fragmentos contextualizados: 800


## Indexación

In [7]:
import os
import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

# ÍNDICE LÉXICO BM25 CON TOKENIZADOR TÉCNICO UNIFICADO
print("Construyendo índice BM25...")
tokenized_corpus = [tokenize_technical_text(doc) for doc in todos_los_chunks]
bm25_index = BM25Okapi(tokenized_corpus)

# CHROMADB PERSISTENTE CON CONTROL DE REINDEXACIÓN
REINDEXAR = True  # Cambiar a False en ejecuciones subsecuentes para no repetir embeddings

chroma_client = chromadb.PersistentClient(path="./chroma_db_ris")
collection_name = "ris_knowledge_base"

if REINDEXAR:
    print("Reindexando base vectorial en ChromaDB...")
    try:
        chroma_client.delete_collection(collection_name)
    except Exception:
        pass

    collection = chroma_client.create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )

    embedding_model = SentenceTransformer("intfloat/multilingual-e5-base")
    passages = [f"passage: {doc}" for doc in todos_los_chunks]
    embeddings = embedding_model.encode(passages, batch_size=32, show_progress_bar=True).tolist()

    collection.add(
        documents=todos_los_chunks,
        embeddings=embeddings,
        metadatas=todos_los_metadatos,
        ids=todos_los_ids
    )
    print(" Indexación completada y guardada en disco.")
else:
    print(" Cargando colección persistente existente de ChromaDB...")
    collection = chroma_client.get_collection(name=collection_name)
    embedding_model = SentenceTransformer("intfloat/multilingual-e5-base")

Construyendo índice BM25...
Reindexando base vectorial en ChromaDB...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

 Indexación completada y guardada en disco.


## Camada de recuperación (RAG)

In [8]:
import numpy as np
from sentence_transformers import CrossEncoder

reranker_model = CrossEncoder("BAAI/bge-reranker-v2-m3")

# Parámetro global calibrable experimentalmente con la matriz de evaluación
THRESHOLD_CONFIANZA = 0.60

def es_respuesta_valida(candidatos, score_top1):
    if not candidatos:
        return False
    if score_top1 < THRESHOLD_CONFIANZA:
        return False
    return True

def search_bm25(query, k=20):
    query_expandida = expandir_query(query)
    tokenized_query = tokenize_technical_text(query_expandida)
    scores = bm25_index.get_scores(tokenized_query)
    top_k_indices = np.argsort(scores)[::-1][:k]

    results = []
    for idx in top_k_indices:
        results.append({
            "id": todos_los_ids[idx],
            "document": todos_los_chunks[idx],
            "metadata": todos_los_metadatos[idx],  # Nombre corregido (sin bug)
            "bm25_score": float(scores[idx])
        })
    return results

def search_vectorial(query, k=20):
    query_text = f"query: {query}"
    query_embedding = embedding_model.encode(query_text).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    vector_results = []
    for i in range(len(results["ids"][0])):
        vector_results.append({
            "id": results["ids"][0][i],
            "document": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "vector_distance": results["distances"][0][i]
        })
    return vector_results

def reciprocal_rank_fusion(bm25_results, vector_results, k_rrf=60, top_n=15):
    rrf_scores = {}
    item_map = {}

    for rank, item in enumerate(bm25_results):
        doc_id = item["id"]
        item_map[doc_id] = item
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_rrf + rank + 1))

    for rank, item in enumerate(vector_results):
        doc_id = item["id"]
        if doc_id not in item_map:
            item_map[doc_id] = item
        rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + (1.0 / (k_rrf + rank + 1))

    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

    fused_list = []
    for doc_id, rrf_score in sorted_docs:
        base_item = item_map[doc_id]
        base_item["rrf_score"] = rrf_score
        fused_list.append(base_item)

    return fused_list

def ejecutar_retrieval_rag(query):
    res_bm25 = search_bm25(query, k=20)
    res_vec = search_vectorial(query, k=20)

    candidatos = reciprocal_rank_fusion(res_bm25, res_vec, top_n=15)

    if not candidatos:
        return [], 0.0, False

    # MEJORA: Evaluamos contra 'texto_raw' para evitar sesgo del encabezado
    pairs = [[query, doc["metadata"]["texto_raw"]] for doc in candidatos]
    raw_scores = reranker_model.predict(pairs)

    scores_norm = 1 / (1 + np.exp(-np.array(raw_scores)))

    for i, doc in enumerate(candidatos):
        doc["cross_encoder_raw"] = float(raw_scores[i])
        doc["confidence_score"] = float(scores_norm[i])

    # Seleccionamos el Top 3 final
    candidatos_ordenados = sorted(candidatos, key=lambda x: x["confidence_score"], reverse=True)[:3]
    top_score = candidatos_ordenados[0]["confidence_score"] if candidatos_ordenados else 0.0

    confiable = es_respuesta_valida(candidatos_ordenados, top_score)
    return candidatos_ordenados, top_score, confiable

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

In [9]:
def construir_prompt_para_llm(pregunta, documentos_recuperados, es_confiable):
    if not es_confiable:
        return None, "No se encontró información técnica suficiente en los manuales ni fichas técnicas para responder a esta consulta con la precisión requerida."

    bloques_contexto = []
    for d in documentos_recuperados:
        meta = d["metadata"]
        ref = f"{meta['tipo_documento']} (Página/Código: {meta.get('pagina', meta.get('codigo'))}, ID: {meta['chunk_id']})"
        bloques_contexto.append(f"--- FUENTE: {ref} ---\n{meta['texto_raw']}")

    contexto_str = "\n\n".join(bloques_contexto)

    prompt = f"""
Eres un asistente técnico especializado en recubrimientos industriales RIS.
Tu tarea es responder la consulta del usuario ÚNICAMENTE utilizando los fragmentos de contexto técnico proporcionados a continuación.

REGLAS DE GENERACIÓN:
1. Responde de forma técnica, precisa y directa.
2. Para cada afirmación importante, indica explícitamente la fuente utilizada (Ejemplo: [Manual Operativo, Página 6]).
3. Si la información solicitada no está explícitamente en el contexto, indica estrictamente: "No dispongo de información suficiente en la documentación para responder a este punto."

CONTEXTO TÉCNICO:
{contexto_str}

PREGUNTA DEL USUARIO:
{pregunta}

RESPUESTA TÉCNICA:
"""
    return prompt, None

In [10]:
pregunta = "¿Cuál es el procedimiento y preparación para corrosión severa con chorro de arena?"

docs, score, es_confiable = ejecutar_retrieval_rag(pregunta)
prompt_final, mensaje_error = construir_prompt_para_llm(pregunta, docs, es_confiable)

print(f"================ RESULTADO RETRIEVAL ================")
print(f"Pregunta: {pregunta}")
print(f"Confianza Top-1: {score:.4f}")
print(f"¿Pasa validación?: {es_confiable}")

if es_confiable:
    print(f"\n================ PROMPT ENVIADO AL LLM ================")
    print(prompt_final)
else:
    print(f"\n================ RESPUESTA RECHAZADA ================")
    print(mensaje_error)

================ RESULTADO RETRIEVAL ================
Pregunta: ¿Cuál es el procedimiento y preparación para corrosión severa con chorro de arena?
Confianza Top-1: 0.7061
¿Pasa validación?: True

================ PROMPT ENVIADO AL LLM ================

Eres un asistente técnico especializado en recubrimientos industriales RIS.
Tu tarea es responder la consulta del usuario ÚNICAMENTE utilizando los fragmentos de contexto técnico proporcionados a continuación.

REGLAS DE GENERACIÓN:
1. Responde de forma técnica, precisa y directa.
2. Para cada afirmación importante, indica explícitamente la fuente utilizada (Ejemplo: [Manual Operativo, Página 6]).
3. Si la información solicitada no está explícitamente en el contexto, indica estrictamente: "No dispongo de información suficiente en la documentación para responder a este punto."

CONTEXTO TÉCNICO:
--- FUENTE: Manual Operativo (Página/Código: 6, ID: manual_operativo_12) ---
METAL LIMPIO SIN ÓXIDO:
Procedimiento: Desengrase → Despolvatado → I

## Producción y validación de respuestas

In [30]:
!pip install -q -U google-genai

In [43]:
import os
import time
import re
import pandas as pd
from google import genai
from google.genai import types

# =====================================================================
# CONFIGURACIÓN DEL CLIENTE
# =====================================================================
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

client = genai.Client(api_key=GEMINI_API_KEY)


# =====================================================================
# PROMPT BUILDER
# =====================================================================
class PromptBuilder:
    @staticmethod
    def construir_prompt(pregunta, documentos_recuperados):
        bloques_contexto = []
        vistos = set()

        for d in documentos_recuperados:
            texto = d["metadata"]["texto_raw"].strip()
            if texto in vistos:
                continue
            vistos.add(texto)

            meta = d["metadata"]
            pag_cod = meta.get('pagina', meta.get('codigo', 'N/A'))
            ref = f"{meta['tipo_documento']}, Página/Código {pag_cod}"
            bloques_contexto.append(f"--- FUENTE [{ref}] ---\n{texto}")

        contexto_unificado = "\n\n".join(bloques_contexto)

        prompt_sistema = """Eres un asistente técnico especializado en recubrimientos industriales RIS.

INSTRUCCIONES DE RIGOR TÉCNICO Y FORMATO:
1. Responde únicamente utilizando la información explícita del CONTEXTO TÉCNICO.
2. Para cada afirmación o sección técnica, DEBES citar la fuente utilizando el formato exacto: [Manual Operativo, Página 6] o [Ficha Técnica, Código MAT-01].
3. Si una respuesta requiere combinar varios fragmentos o fuentes, hazlo explícitamente indicando todas las fuentes utilizadas para esa sección.
4. PROHIBIDO UTILIZAR FORMATO LATEX O FÓRMULAS MATEMÁTICAS (No uses $$ o $ para procedimientos ni flechas).
5. Presenta las secuencias, pasos o parámetros mediante listas numeradas o con viñetas en Markdown estándar.
6. Si la información no aparece en el contexto, responde estrictamente: "No dispongo de información suficiente en la documentación técnica para responder a esta pregunta."
7. No utilices conocimiento previo ni asumas datos no documentados."""

        prompt_usuario = f"""CONTEXTO TÉCNICO:
{contexto_unificado}

CONSULTA DEL USUARIO:
{pregunta}

RESPUESTA TÉCNICA:"""

        return prompt_sistema, prompt_usuario, contexto_unificado


# =====================================================================
# LLM GENERATOR
# =====================================================================
class LLMGenerator:
    def __init__(self, client_instance, modelo_fijo="gemini-3.5-flash"):
        self.client = client_instance
        self.modelo = modelo_fijo

    def generar(self, prompt_sistema, prompt_usuario, max_intentos=2):
        for intento in range(max_intentos):
            try:
                response = self.client.models.generate_content(
                    model=self.modelo,
                    contents=prompt_usuario,
                    config=types.GenerateContentConfig(
                        system_instruction=prompt_sistema,
                        temperature=0.0
                    )
                )
                if response.text:
                    return response.text.strip(), "ok"

            except Exception as e:
                msg = str(e)
                if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                    tiempo_espera = 10 * (intento + 1)
                    print(f"⚠️ Ráfaga/Cuota temporal (429). Reintentando en {tiempo_espera}s...")
                    time.sleep(tiempo_espera)
                else:
                    print(f"⚠️ Error en modelo {self.modelo}: {msg}")
                    break

        return None, "error_api_o_cuota"


# =====================================================================
# VALIDADOR TÉCNICO
# =====================================================================
class ResponseValidator:
    @staticmethod
    def validar(respuesta_llm, es_confiable_retrieval):
        if not es_confiable_retrieval:
            return False, "retrieval_bajo_umbral"

        if len(respuesta_llm.strip()) < 30:
            return False, "respuesta_demasiado_corta"

        frases_evasivas = [
            "no dispongo de información",
            "no se encuentra en los documentos",
            "información insuficiente"
        ]
        if any(f in respuesta_llm.lower() for f in frases_evasivas):
            return False, "informacion_insuficiente"

        # Verifica la presencia de citas
        tiene_citas = bool(re.search(r"\[.*?(Página|Código|Pág|MAT|EQ|Manual|Ficha).*?\]", respuesta_llm, re.IGNORECASE))
        if not tiene_citas:
            return False, "sin_citas_validas"

        return True, "ok"


# =====================================================================
# FORMATTER CON NIVEL DE CONFIANZA Y FUENTES LIMPIAS
# =====================================================================
class ResponseFormatter:
    @staticmethod
    def obtener_nivel_confianza(score):
        if score >= 0.80:
            return f"Alta ({score:.2f})"
        elif score >= 0.60:
            return f"Media ({score:.2f})"
        else:
            return f"Baja ({score:.2f})"

    @staticmethod
    def generar_fallback(pregunta, motivo):
        mensajes = {
            "score_retrieval_bajo": "No se encontraron documentos técnicos con suficiente evidencia en la base de conocimientos para responder esta consulta.",
            "error_api_o_cuota": "La consulta no pudo ser procesada por limitaciones temporales en la API del LLM. Intente de nuevo en un minuto.",
            "sin_citas_validas": "La respuesta generada no incluyó referencias explícitas a la documentación.",
            "informacion_insuficiente": "La información solicitada no está disponible en la documentación técnica actual."
        }
        explicacion = mensajes.get(motivo, "No fue posible procesar la consulta con suficiente evidencia documental.")

        return f"""### ⚠️ Consulta No Completada

> **Consulta:** *"{pregunta}"*

{explicacion}

**Sugerencias:**
• Revisa el manual o ficha técnica correspondiente.
• Consulte directamente con el área técnica responsable.
• Reformule la pregunta con términos más específicos.

---
*Estado de Seguridad:* Rechazado (`Motivo: {motivo}`)"""

    @classmethod
    def dar_formato_final(cls, respuesta_llm, documentos_recuperados, score_retrieval):
        # Extraer fuentes únicas utilizadas sin corchetes repetidos
        citas_encontradas = re.findall(r'\[(.*?)\]', respuesta_llm)
        fuentes_limpias = set()

        for cita in citas_encontradas:
            if any(k in cita.lower() for k in ['manual', 'ficha', 'página', 'código', 'pág']):
                fuentes_limpias.add(f"• {cita.strip()}")

        if not fuentes_limpias:
            for d in documentos_recuperados:
                meta = d["metadata"]
                pag_cod = meta.get('pagina', meta.get('codigo', 'N/A'))
                fuentes_limpias.add(f"• {meta['tipo_documento']}, Página/Código {pag_cod}")

        bloque_fuentes = "\n".join(sorted(list(fuentes_limpias)))
        nivel_confianza = cls.obtener_nivel_confianza(score_retrieval)

        return f"""{respuesta_llm}

---
### 📚 Fuentes Consultadas
{bloque_fuentes}

**Nivel de Confianza del Retrieval:** `{nivel_confianza}`"""


# =====================================================================
# AGENTE RAG CON TRAZABILIDAD DE CHUNKS EN EXPERIMENTOS
# =====================================================================
class RAGAgent:
    def __init__(self):
        self.prompt_builder = PromptBuilder()
        self.llm_generator = LLMGenerator(client_instance=client, modelo_fijo="gemini-3.5-flash")
        self.validator = ResponseValidator()
        self.formatter = ResponseFormatter()
        self.historial_evaluacion = []

    def responder_consulta(self, pregunta):
        inicio_tiempo = time.time()

        # Retrieval
        candidatos, score, es_confiable = ejecutar_retrieval_rag(pregunta)

        # Extraer IDs/Trazabilidad de chunks
        ids_chunks = [
            f"{d['metadata'].get('tipo_documento','doc')}_p{d['metadata'].get('pagina', d['metadata'].get('codigo', 'NA'))}"
            for d in candidatos
        ]

        if not es_confiable:
            respuesta_final = self.formatter.generar_fallback(pregunta, "score_retrieval_bajo")
            self._registrar_experimento(pregunta, score, "rechazado_retrieval", ids_chunks, time.time() - inicio_tiempo)
            return respuesta_final

        # 2. Prompting
        prompt_sis, prompt_usr, contexto_txt = self.prompt_builder.construir_prompt(pregunta, candidatos)

        # 3. LLM Generation
        respuesta_raw, estatus_llm = self.llm_generator.generar(prompt_sis, prompt_usr)

        if estatus_llm != "ok":
            respuesta_final = self.formatter.generar_fallback(pregunta, estatus_llm)
            self._registrar_experimento(pregunta, score, f"fallo_llm_{estatus_llm}", ids_chunks, time.time() - inicio_tiempo)
            return respuesta_final

        # 4. Validation
        es_valida, motivo = self.validator.validar(respuesta_raw, es_confiable)

        # 5. Output
        if es_valida:
            respuesta_final = self.formatter.dar_formato_final(respuesta_raw, candidatos, score)
            self._registrar_experimento(pregunta, score, "éxito", ids_chunks, time.time() - inicio_tiempo)
            return respuesta_final
        else:
            respuesta_final = self.formatter.generar_fallback(pregunta, motivo)
            self._registrar_experimento(pregunta, score, f"rechazado_validación_{motivo}", ids_chunks, time.time() - inicio_tiempo)
            return respuesta_final

    def _registrar_experimento(self, pregunta, score, estado, ids_chunks, tiempo_ejecucion):
        registro = {
            "Fecha": time.strftime("%Y-%m-%d %H:%M:%S"),
            "Modelo_LLM": self.llm_generator.modelo,
            "Pregunta": pregunta,
            "Score_Retrieval": round(score, 4),
            "Nivel_Confianza": self.formatter.obtener_nivel_confianza(score),
            "Chunks_Consultados": ", ".join(ids_chunks),
            "Estado": estado,
            "Tiempo_Segundos": round(tiempo_ejecucion, 2)
        }
        self.historial_evaluacion.append(registro)

    def exportar_historial(self, ruta_csv="historial_evaluacion_rag.csv"):
        df = pd.DataFrame(self.historial_evaluacion)
        df.to_csv(ruta_csv, index=False)
        print(f"📊 Historial guardado exitosamente en: {ruta_csv}")

In [44]:
agente_ris = RAGAgent()

# Test 1: Consulta dentro de dominio
print("================ TEST 1: CONSULTA DENTRO DE DOMINIO ================\n")
pregunta_1 = "¿Cuál es el procedimiento y preparación para corrosión severa con chorro de arena?"
print(agente_ris.responder_consulta(pregunta_1))

print("\n" + "="*70 + "\n")

# Test 2: Consulta fuera de dominio
print("================ TEST 2: CONSULTA FUERA DE DOMINIO ================\n")
pregunta_2 = "¿Cuál es el procedimiento para solicitar viáticos de viaje?"
print(agente_ris.responder_consulta(pregunta_2))

================ TEST 1: CONSULTA DENTRO DE DOMINIO ================

De acuerdo con la documentación técnica, el procedimiento y los parámetros para la preparación de una superficie con corrosión severa mediante chorro de arena son los siguientes:

**Procedimiento paso a paso:**
1. Desengrase
2. Chorro de Arena (SA 2.5)
3. Lijado
4. Despolvatado
5. Inspección

**Parámetros del proceso:**
* **Tiempo estimado:** 24-48 horas
* **Rugosidad target (objetivo):** Ra 4.5-6.5 µm

[Manual Operativo, Página/Código 6]

---
### 📚 Fuentes Consultadas
• Manual Operativo, Página/Código 6

**Nivel de Confianza del Retrieval:** `Media (0.71)`


================ TEST 2: CONSULTA FUERA DE DOMINIO ================

### ⚠️ Consulta No Completada

> **Consulta:** *"¿Cuál es el procedimiento para solicitar viáticos de viaje?"*

No se encontraron documentos técnicos con suficiente evidencia en la base de conocimientos para responder esta consulta.

**Sugerencias:**
• Revisa el manual o ficha técnica correspon